## Übung: 3 VMs zu einem Kubernetes-Cluster verbinden

MicroK8s bietet eine einfache und effiziente Möglichkeit, Kubernetes-Cluster lokal oder in produktionsnahen Umgebungen zu betreiben.

Das Hinzufügen von Nodes zu einem bestehenden MicroK8s-Cluster erfolgt mit dem Befehl `microk8s add-node`.

Dieser Befehl generiert die notwendigen Verbindungsinformationen und Tokens, damit neue Nodes, zum Beispiel Control-Plane- oder Worker-Nodes, sicher in den Cluster integriert werden können.

---

Um die Vorgänge zu visualisieren, verwenden wir Headlamp. Navigiert darin zu **Cluster → Nodes**.


In [ ]:
%%bash
echo "HeadLamp: http://"$(cat ~/data/server-ip)":30444"
kubectl create token headlamp-admin -n kube-system --duration=48h

**Vorbereiten der Worker Nodes**

microk8s Kubernetes auf den Worker Nodes installieren

In [ ]:
%%bash
ssh worker-01-default -- sudo snap install microk8s --classic

In [ ]:
%%bash
ssh worker-02-default -- sudo snap install microk8s --classic

Ubuntu 26.xx Hack (altes sudo verwenden)

In [ ]:
%%bash
ssh worker-01-default -- sudo update-alternatives --set sudo /usr/bin/sudo.ws 

In [ ]:
%%bash
ssh worker-02-default -- sudo update-alternatives --set sudo /usr/bin/sudo.ws 

**Worker Nodes mit der Control Plane Node verbinden (joinen)**

In [ ]:
%%bash
# Verbindungsinformationen und Token aufbereiten
export JOIN=$(sudo microk8s add-node | grep worker | tail -1)
echo $JOIN
# Knoten (Worker) 01 integrieren
ssh worker-01-default -- sudo ${JOIN}

In [ ]:
%%bash
# Knoten (Worker) 02 integrieren
export JOIN=$(sudo microk8s add-node | grep worker | tail -1)
ssh worker-02-default -- sudo ${JOIN}

In [ ]:
! kubectl get nodes 

- - -
### Zentraler persistenter Speicher

Damit alle Kubernetes-Nodes, sowohl Control Plane als auch Worker, denselben zentralen persistenten Speicher verwenden können, wurde während der Installation auf der Control Plane ein NFS-Share unter `/data` eingerichtet.


In [ ]:
! cat /etc/exports

Die Worker-Nodes müssen sich mit diesem NFS-Share verbinden, damit sie auf denselben persistenten Speicher zugreifen können.

In [ ]:
! ssh worker-01-default -- "sudo mount -t nfs control-01-default:/data /data; df -h | grep /data"

In [ ]:
! ssh worker-02-default -- "sudo mount -t nfs control-01-default:/data /data; df -h | grep /data"

Die entsprechende Dateiablage, in Kubernetes als PersistentVolume (PV) bezeichnet, wurde während der Kubernetes-Installation eingerichtet. Das PersistentVolume verweist über den Eintrag `hostPath` auf das Verzeichnis `/data`.

Diese Konfiguration trennt den Speicher vom Container- beziehungsweise Pod-Lebenszyklus. Dadurch bleiben Daten auch nach Neustarts oder Löschungen von Containern und Pods erhalten.

In [ ]:
%%bash
kubectl get pv -o go-template='{{ range .items }}
apiVersion: v1
kind: PersistentVolume
metadata:
  name: {{ .metadata.name }}
spec:
  capacity:
    storage: {{ .spec.capacity.storage }}
  accessModes:
  {{- range .spec.accessModes }}
    - {{ . }}
  {{- end }}
  storageClassName: {{ .spec.storageClassName }}
  persistentVolumeReclaimPolicy: {{ .spec.persistentVolumeReclaimPolicy }}
  volumeMode: {{ .spec.volumeMode }}

  {{- if .spec.hostPath }}
  hostPath:
    path: {{ .spec.hostPath.path }}
    type: {{ .spec.hostPath.type }}
  {{- end }}

  {{- if .spec.claimRef }}
  claimRef:
    namespace: {{ .spec.claimRef.namespace }}
    name: {{ .spec.claimRef.name }}
  {{- end }}
status:
  phase: {{ .status.phase }}
---
{{ end }}'
